In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

# helper func
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def relu(z):
    return np.maximum(0, z)
def relu_derivative(z):
    return (z > 0).astype(float)

def compute_loss(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-8, 1 - 1e-8)
    loss = -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    return loss

def compute_accuracy(y_true, y_pred):
    #if  pred >= 0.5 then class 1, else class 0
    predicted_class = (y_pred >= 0.5).astype(int)
    return np.mean(predicted_class == y_true)


N = 3000

x1 = np.random.uniform(-2, 2, N)
x2 = np.random.uniform(-2, 2, N)

y = (x1**2 + x2**2 > 1.5).astype(int)

X = np.column_stack([x1, x2])
idx   = np.random.permutation(N)
t_end = int(0.70 * N)   # 2100
v_end = int(0.85 * N)   # 2550

X_train = X[idx[:t_end]]       ;   y_train = y[idx[:t_end]]
X_val   = X[idx[t_end:v_end]]  ;   y_val   = y[idx[t_end:v_end]]
X_test  = X[idx[v_end:]]       ;   y_test  = y[idx[v_end:]]

print(f"\nDataset created:")
print(f"  Train samples : {X_train.shape[0]}")
print(f"  Val   samples : {X_val.shape[0]}")
print(f"  Test  samples : {X_test.shape[0]}")

#  3 arch
# [2, 8, 1] means: 2 inputs, 8 hidden neurons, 1 output
arch_2layer  = [2, 8, 1]
arch_5layer  = [2, 8, 8, 8, 8, 1]
arch_10layer = [2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1]

def count_and_print_params(arch, name):
    print(f"\n{name} - Architecture: {arch}")
    total = 0
    for i in range(1, len(arch)):
        n_in  = arch[i-1]
        n_out = arch[i]
        layer_params = (n_in * n_out) + n_out
        print(f"  Layer {i}: ({n_in} x {n_out}) + {n_out} = {layer_params}")
        total += layer_params
    print(f"  total parameter = {total}")
    return total

p_2layer  = count_and_print_params(arch_2layer,  "2-layer  network")
p_5layer  = count_and_print_params(arch_5layer,  "5-layer  network")
p_10layer = count_and_print_params(arch_10layer, "10-layer network")


def initialize_weights(arch):
    params = {}
    for i in range(1, len(arch)):
        n_in  = arch[i-1]
        n_out = arch[i]
        params["W" + str(i)] = np.random.randn(n_in, n_out) * np.sqrt(2.0 / n_in)
        params["b" + str(i)] = np.zeros((1, n_out))
    return params

def forward_pass(X, params, activation="relu"):
    cache = {}
    cache["A0"] = X
    num_layers = len(params) // 2

    for i in range(1, num_layers):
        # linear step
        Z = cache["A" + str(i-1)] @ params["W" + str(i)] + params["b" + str(i)]
        cache["Z" + str(i)] = Z

        # activation step for hidden layers
        if activation == "relu":
            cache["A" + str(i)] = relu(Z)
        else:
            cache["A" + str(i)] = sigmoid(Z)

    Z_last = cache["A" + str(num_layers-1)] @ params["W" + str(num_layers)] + params["b" + str(num_layers)]
    cache["Z" + str(num_layers)] = Z_last
    cache["A" + str(num_layers)] = sigmoid(Z_last)

    return cache


#  backward pass
def backward_pass(X, y, params, cache, activation="relu"):
    grads = {}
    N = X.shape[0]
    num_layers = len(params) // 2

    dA = cache["A" + str(num_layers)] - y.reshape(-1, 1)
    for i in range(num_layers, 0, -1):
        A_prev = cache["A" + str(i-1)]
        grads["W" + str(i)] = (A_prev.T @ dA) / N
        grads["b" + str(i)] = np.mean(dA, axis=0, keepdims=True)

        if i > 1:
            dA = dA @ params["W" + str(i)].T
            if activation == "relu":
                dA = dA * relu_derivative(cache["Z" + str(i-1)])
            else:
                s = cache["A" + str(i-1)]
                dA = dA * (s * (1 - s))

    return grads

def get_gradient_norms(grads, num_layers):
    norms = []
    for i in range(1, num_layers + 1):
        norm = np.sqrt(np.sum(grads["W" + str(i)] ** 2))
        norms.append(norm)
    return norms

# SGD
def sgd_update(params, grads, lr=0.01):
    for key in params:
        params[key] = params[key] - lr * grads[key]
    return params

def momentum_update(params, grads, velocity, lr=0.01, beta=0.9):
    for key in params:
        velocity[key] = beta * velocity.get(key, 0) + lr * grads[key]
        params[key]   = params[key] - velocity[key]
    return params, velocity

def train_dense_network(X_tr, y_tr, X_v, y_v, arch,
                        activation="relu", optimizer="sgd",
                        epochs=300, lr=0.01):

    params     = initialize_weights(arch)
    velocity   = {}
    num_layers = len(arch) - 1

    history = {
        "train_loss": [], "val_loss": [],
        "train_acc":  [], "val_acc":  [],
        "grad_norms": []
    }

    for epoch in range(epochs):

        # forward pass on training data
        cache = forward_pass(X_tr, params, activation)

        y_pred_train = cache["A" + str(num_layers)].flatten()

        # training loss and accuracy
        t_loss = compute_loss(y_tr, y_pred_train)
        t_acc  = compute_accuracy(y_tr, y_pred_train)

        #  backward pass
        grads = backward_pass(X_tr, y_tr, params, cache, activation)

        #  Save gradient norms
        history["grad_norms"].append(get_gradient_norms(grads, num_layers))

        #  update weights using chosen optimizer
        if optimizer == "sgd":
            params = sgd_update(params, grads, lr)
        else:
            params, velocity = momentum_update(params, grads, velocity, lr)

        val_cache = forward_pass(X_v, params, activation)
        y_pred_val = val_cache["A" + str(num_layers)].flatten()
        v_loss = compute_loss(y_v, y_pred_val)
        v_acc  = compute_accuracy(y_v, y_pred_val)


        history["train_loss"].append(t_loss)
        history["val_loss"].append(v_loss)
        history["train_acc"].append(t_acc)
        history["val_acc"].append(v_acc)

    return params, history


def evaluate_on_test(X_te, y_te, params, activation, arch):
    num_layers = len(arch) - 1
    cache  = forward_pass(X_te, params, activation)
    y_pred = cache["A" + str(num_layers)].flatten()
    return compute_loss(y_te, y_pred), compute_accuracy(y_te, y_pred)


def save_plot(history, title, filename):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].plot(history["train_loss"], label="Train Loss")
    axes[0].plot(history["val_loss"],   label="Val Loss")
    axes[0].set_title(title + " - Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(history["train_acc"], label="Train Accuracy")
    axes[1].plot(history["val_acc"],   label="Val Accuracy")
    axes[1].set_title(title + " - Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.savefig(filename, dpi=80)
    plt.close()

architectures = {
    "2layer":  arch_2layer,
    "5layer":  arch_5layer,
    "10layer": arch_10layer,
}
param_counts = {
    "2layer": p_2layer,
    "5layer": p_5layer,
    "10layer": p_10layer,
}

master_results = []
saved_histories = {}
for arch_name, arch in architectures.items():
    for act in ["sigmoid", "relu"]:
        for opt in ["sgd", "momentum"]:

            label = f"{arch_name}_{act}_{opt}"
            print(f"  Training: {label}")

            trained_params, hist = train_dense_network(
                X_train, y_train, X_val, y_val,
                arch, activation=act, optimizer=opt, epochs=300
            )

            test_loss, test_acc = evaluate_on_test(
                X_test, y_test, trained_params, act, arch
            )

            saved_histories[label] = hist
            save_plot(hist, label, f"part1_{label}.png")


            master_results.append({
                "Model":      arch_name,
                "Depth":      len(arch) - 1,
                "Activation": act,
                "Optimizer":  opt,
                "Parameters": param_counts[arch_name],
                "Train Acc":  round(hist["train_acc"][-1], 4),
                "Val Acc":    round(hist["val_acc"][-1],   4),
                "Test Acc":   round(test_acc, 4),
            })


h_2layer  = saved_histories["2layer_sigmoid_sgd"]
h_10layer = saved_histories["10layer_sigmoid_sgd"]

norms_2  = np.array(h_2layer["grad_norms"])
norms_10 = np.array(h_10layer["grad_norms"])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(norms_2[:, 0],  label="Layer 1 (first)")
axes[0].plot(norms_2[:, -1], label="Layer 2 (last)")
axes[0].set_title("Gradient Norms - 2 Layer Sigmoid")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Frobenius Norm")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(norms_10[:, 0],  label="Layer 1 (first)")
axes[1].plot(norms_10[:, -1], label="Layer 10 (last)")
axes[1].set_title("Gradient Norms - 10 Layer Sigmoid")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Frobenius Norm")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig("part1_gradient_norms.png", dpi=80)
plt.close()

print("\n-table for part 1 ")
print(f"{'Model':<10} {'D':<4} {'Act':<10} {'Opt':<10} {'Params':<8} {'TrainAcc':<10} {'ValAcc':<10} {'TestAcc'}")
print("-" * 72)
for r in master_results:
    print(f"{r['Model']:<10} {r['Depth']:<4} {r['Activation']:<10} {r['Optimizer']:<10} "
          f"{r['Parameters']:<8} {r['Train Acc']:<10} {r['Val Acc']:<10} {r['Test Acc']}")

# image dataset
def generate_image_dataset(N=3000):
    images = []
    labels = []

    for _ in range(N):
        img   = np.zeros((8, 8))
        label = np.random.randint(0, 2)

        if label == 0:
            img[:, 3] = 1.0
            img[:, 4] = 1.0
        else:
            img[3, :] = 1.0
            img[4, :] = 1.0


        img = img + np.random.normal(0, 0.1, (8, 8))
        images.append(img)
        labels.append(label)

    return np.array(images), np.array(labels)


images, labels = generate_image_dataset(3000)

# 70 - 15 - 15
idx2  = np.random.permutation(3000)
te2   = int(0.70 * 3000)
ve2   = int(0.85 * 3000)

Xi_train = images[idx2[:te2]]      ;   yi_train = labels[idx2[:te2]]
Xi_val   = images[idx2[te2:ve2]]   ;   yi_val   = labels[idx2[te2:ve2]]
Xi_test  = images[idx2[ve2:]]      ;   yi_test  = labels[idx2[ve2:]]

print(f"\nimage dataset created: Train={Xi_train.shape}, Val={Xi_val.shape}, Test={Xi_test.shape}")

Xf_train = Xi_train.reshape(-1, 64)
Xf_val   = Xi_val.reshape(-1,   64)
Xf_test  = Xi_test.reshape(-1,  64)

dense_img_arch = [64, 32, 1]
print(f"arch: {dense_img_arch}")
p_dense_img = 0
for i in range(1, len(dense_img_arch)):
    p = dense_img_arch[i-1] * dense_img_arch[i] + dense_img_arch[i]
    print(f"  Layer {i}: ({dense_img_arch[i-1]} x {dense_img_arch[i]}) + {dense_img_arch[i]} = {p}")
    p_dense_img += p
print(f"  dense baseline total params = {p_dense_img}")

dense_img_params, h_dense_img = train_dense_network(
    Xf_train, yi_train, Xf_val, yi_val,
    dense_img_arch, activation="relu", optimizer="sgd", epochs=200, lr=0.01
)

_, dense_test_acc = evaluate_on_test(Xf_test, yi_test, dense_img_params, "relu", dense_img_arch)

save_plot(h_dense_img, "Dense Baseline", "part2_dense_baseline.png")
print(f"  Train={h_dense_img['train_acc'][-1]:.4f}  Val={h_dense_img['val_acc'][-1]:.4f}  Test={dense_test_acc:.4f}")

#forward
def conv_forward(X, kernel, bias):
    N, H, W    = X.shape
    F          = kernel.shape[0]
    num_filters = kernel.shape[2]
    out_H = H - F + 1
    out_W = W - F + 1

    output = np.zeros((N, out_H, out_W, num_filters))

    for i in range(out_H):
        for j in range(out_W):
            patch = X[:, i:i+F, j:j+F]

            for f in range(num_filters):
                output[:, i, j, f] = np.sum(patch * kernel[:, :, f], axis=(1, 2)) + bias[f]

    return output

# backward
def conv_backward(d_output, X, kernel):
    N, H, W     = X.shape
    F            = kernel.shape[0]
    num_filters  = kernel.shape[2]
    out_H, out_W = d_output.shape[1], d_output.shape[2]

    d_kernel = np.zeros_like(kernel)
    d_bias   = np.zeros(num_filters)
    dX       = np.zeros_like(X)

    for i in range(out_H):
        for j in range(out_W):
            patch = X[:, i:i+F, j:j+F]
            for f in range(num_filters):
                d = d_output[:, i, j, f]
                d_kernel[:, :, f] += np.sum(patch * d[:, None, None], axis=0)
                d_bias[f]         += np.sum(d)
                dX[:, i:i+F, j:j+F] += kernel[:, :, f] * d[:, None, None]

    return dX, d_kernel / N, d_bias / N


def maxpool_forward(X, pool_size=2):
    N, H, W, C = X.shape
    pH = H // pool_size
    pW = W // pool_size

    output = np.zeros((N, pH, pW, C))
    mask   = np.zeros_like(X)

    for i in range(pH):
        for j in range(pW):
            region = X[:, i*pool_size:(i+1)*pool_size,
                          j*pool_size:(j+1)*pool_size, :]   # (N, 2, 2, C)
            output[:, i, j, :] = np.max(region, axis=(1, 2))
            mask[:, i*pool_size:(i+1)*pool_size,
                    j*pool_size:(j+1)*pool_size, :] = (region == output[:, i:i+1, j:j+1, :])

    return output, mask


def maxpool_backward(d_output, mask, pool_size=2):
    N, pH, pW, C = d_output.shape
    dX = np.zeros_like(mask)

    for i in range(pH):
        for j in range(pW):
            dX[:, i*pool_size:(i+1)*pool_size,
                   j*pool_size:(j+1)*pool_size, :] = \
                d_output[:, i:i+1, j:j+1, :] * mask[:, i*pool_size:(i+1)*pool_size,
                                                         j*pool_size:(j+1)*pool_size, :]
    return dX

"""# task 2c"""

F         = 3          # filter size 3x3
Cout      = 4          # no. of filters
pool_flat = 3 * 3 * 4  # after conv(6x6) + pool(3x3) with 4 filters = 36

cnn_conv_params  = (F * F * Cout) + Cout
cnn_dense_params = pool_flat * 1 + 1
cnn_total_params = cnn_conv_params + cnn_dense_params

print(f"\n  dense Network [64->32->1]:")
print(f"    layer 1: (64 x 32) + 32 = {64*32+32}")
print(f"    layer 2: (32 x 1)  + 1  = {32*1+1}")

print(f"   total = {64*32+32 + 32*1+1}")
print(f"\n  CNN [Conv(3x3 x4 filters) -> Pool -> Output]:")

print(f"    Conv layer : ({F}x{F}x{Cout}) + {Cout} = {cnn_conv_params}")
print(f"    Dense layer: ({pool_flat}x1) + 1 = {cnn_dense_params}")
print(f"    TOTAL = {cnn_total_params}")
print(f"\n  CNN uses {p_dense_img - cnn_total_params} ")



def train_cnn(Xi_tr, yi_tr, Xi_v, yi_v,
              use_pooling=True, dropout_rate=0.0,
              optimizer="sgd", epochs=150, lr=0.01):

    F, num_filters = 3, 4

    flat_size = 36 if use_pooling else 144

    # initializing CNN weights
    kernel = np.random.randn(F, F, num_filters) * 0.1
    bias_c = np.zeros(num_filters)
    W_out  = np.random.randn(flat_size, 1) * np.sqrt(2.0 / flat_size)
    b_out  = np.zeros((1, 1))

    # momentum velocity
    vel_kernel = np.zeros_like(kernel)
    vel_biasc  = np.zeros_like(bias_c)
    vel_Wout   = np.zeros_like(W_out)
    vel_bout   = np.zeros_like(b_out)

    # adam moment estimates
    m_k = np.zeros_like(kernel);  v_k = np.zeros_like(kernel)
    m_b = np.zeros_like(bias_c);  v_b = np.zeros_like(bias_c)
    m_W = np.zeros_like(W_out);   v_W = np.zeros_like(W_out)
    m_o = np.zeros_like(b_out);   v_o = np.zeros_like(b_out)

    history = {"train_loss": [], "val_loss": [],
               "train_acc":  [], "val_acc":  []}

    for epoch in range(1, epochs + 1):
        N = Xi_tr.shape[0]

        conv_out = conv_forward(Xi_tr, kernel, bias_c)
        relu_out = relu(conv_out)

        if use_pooling:
            pool_out, pool_mask = maxpool_forward(relu_out, pool_size=2)
            flat = pool_out.reshape(N, -1)
        else:
            flat = relu_out.reshape(N, -1)
            pool_out = None
            pool_mask = None

        if dropout_rate > 0.0:
            keep = 1.0 - dropout_rate
            drop_mask = (np.random.rand(*flat.shape) < keep) / keep
            flat = flat * drop_mask
        else:
            drop_mask = np.ones_like(flat)

        # dense output layer
        z_out  = flat @ W_out + b_out
        y_pred = sigmoid(z_out).flatten()

        history["train_loss"].append(compute_loss(yi_tr, y_pred))
        history["train_acc"].append(compute_accuracy(yi_tr, y_pred))

        # backward pass
        dz     = (y_pred - yi_tr).reshape(-1, 1) / N
        dW_out = flat.T @ dz
        db_out = np.sum(dz, axis=0, keepdims=True)

        d_flat = (dz @ W_out.T) * drop_mask
        if use_pooling:
            d_relu = maxpool_backward(d_flat.reshape(pool_out.shape), pool_mask)
        else:
            d_relu = d_flat.reshape(relu_out.shape)

        d_conv = d_relu * (conv_out > 0)

        _, d_kernel, d_biasc = conv_backward(d_conv, Xi_tr, kernel)

        # weight update

        if optimizer == "sgd":
            kernel -= lr * d_kernel
            bias_c -= lr * d_biasc
            W_out  -= lr * dW_out
            b_out  -= lr * db_out

        elif optimizer == "momentum":
            beta = 0.9
            vel_kernel = beta * vel_kernel + lr * d_kernel ;  kernel -= vel_kernel
            vel_biasc  = beta * vel_biasc  + lr * d_biasc  ;  bias_c -= vel_biasc
            vel_Wout   = beta * vel_Wout   + lr * dW_out   ;  W_out  -= vel_Wout
            vel_bout   = beta * vel_bout   + lr * db_out   ;  b_out  -= vel_bout
        elif optimizer == "adam":
            b1, b2, eps_adam, lr_adam = 0.9, 0.999, 1e-8, 0.001

            # Adam update formula for one parameter
            def adam_update(param, grad, m, v, t):
                m = b1 * m + (1 - b1) * grad
                v = b2 * v + (1 - b2) * grad ** 2
                m_hat = m / (1 - b1 ** t)   # bias correction
                v_hat = v / (1 - b2 ** t)   # bias correction
                param = param - lr_adam * m_hat / (np.sqrt(v_hat) + eps_adam)
                return param, m, v

            kernel, m_k, v_k = adam_update(kernel, d_kernel, m_k, v_k, epoch)
            bias_c, m_b, v_b = adam_update(bias_c, d_biasc,  m_b, v_b, epoch)
            W_out,  m_W, v_W = adam_update(W_out,  dW_out,   m_W, v_W, epoch)
            b_out,  m_o, v_o = adam_update(b_out,  db_out,   m_o, v_o, epoch)

        cv   = conv_forward(Xi_v, kernel, bias_c)
        rv   = relu(cv)
        if use_pooling:
            pv, _ = maxpool_forward(rv, pool_size=2)
            fv    = pv.reshape(Xi_v.shape[0], -1)
        else:
            fv    = rv.reshape(Xi_v.shape[0], -1)
        yv = sigmoid(fv @ W_out + b_out).flatten()
        history["val_loss"].append(compute_loss(yi_v, yv))
        history["val_acc"].append(compute_accuracy(yi_v, yv))

    # Return weights so we can predict on test set
    weights = (kernel, bias_c, W_out, b_out)
    return weights, history

# helper to predict using trained CNN weights
def cnn_predict(Xi, weights, use_pooling=True):
    kernel, bias_c, W_out, b_out = weights
    cv = conv_forward(Xi, kernel, bias_c)
    rv = relu(cv)
    if use_pooling:
        pv, _ = maxpool_forward(rv, pool_size=2)
        fv    = pv.reshape(Xi.shape[0], -1)
    else:
        fv = rv.reshape(Xi.shape[0], -1)
    return sigmoid(fv @ W_out + b_out).flatten()

w_pool,   h_pool   = train_cnn(Xi_train, yi_train, Xi_val, yi_val,
                                use_pooling=True,  dropout_rate=0.0)
acc_pool   = compute_accuracy(yi_test, cnn_predict(Xi_test, w_pool,  True))

w_nopool, h_nopool = train_cnn(Xi_train, yi_train, Xi_val, yi_val,
                                use_pooling=False, dropout_rate=0.0)
acc_nopool = compute_accuracy(yi_test, cnn_predict(Xi_test, w_nopool, False))

w_drop,   h_drop   = train_cnn(Xi_train, yi_train, Xi_val, yi_val,
                                use_pooling=True,  dropout_rate=0.3)
acc_drop   = compute_accuracy(yi_test, cnn_predict(Xi_test, w_drop,  True))

print(f"\n  Results:")

print(f"  With Pooling    -> Val={h_pool['val_acc'][-1]:.4f}    Test={acc_pool:.4f}")
print(f"  Without Pooling -> Val={h_nopool['val_acc'][-1]:.4f}  Test={acc_nopool:.4f}")
print(f"  With Dropout    -> Val={h_drop['val_acc'][-1]:.4f}    Test={acc_drop:.4f}")

# Save plots
for h, name, fname in [
    (h_pool,   "CNN with Pooling",    "part2_cnn_pool.png"),
    (h_nopool, "CNN without Pooling", "part2_cnn_nopool.png"),
    (h_drop,   "CNN with Dropout",    "part2_cnn_dropout.png"),
]:
    save_plot(h, name, fname)

w_sgd,  h_sgd  = train_cnn(Xi_train, yi_train, Xi_val, yi_val,
                             optimizer="sgd",      epochs=150, lr=0.01)

w_mom,  h_mom  = train_cnn(Xi_train, yi_train, Xi_val, yi_val,
                             optimizer="momentum", epochs=150, lr=0.01)

w_adam, h_adam = train_cnn(Xi_train, yi_train, Xi_val, yi_val,
                             optimizer="adam",     epochs=150, lr=0.001)

acc_sgd  = compute_accuracy(yi_test, cnn_predict(Xi_test, w_sgd,  True))
acc_mom  = compute_accuracy(yi_test, cnn_predict(Xi_test, w_mom,  True))
acc_adam = compute_accuracy(yi_test, cnn_predict(Xi_test, w_adam, True))

print(f"\n  SGD      -> Val={h_sgd['val_acc'][-1]:.4f}   Test={acc_sgd:.4f}")
print(f"  Momentum -> Val={h_mom['val_acc'][-1]:.4f}   Test={acc_mom:.4f}")
print(f"  Adam     -> Val={h_adam['val_acc'][-1]:.4f}   Test={acc_adam:.4f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for h, label in [(h_sgd, "SGD"), (h_mom, "Momentum"), (h_adam, "Adam")]:
    axes[0].plot(h["val_loss"], label=label)
    axes[1].plot(h["val_acc"],  label=label)

axes[0].set_title("CNN Validation Loss - Optimizer Comparison")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)
axes[1].set_title("CNN Validation Accuracy - Optimizer Comparison")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig("part3_optimizer_comparison.png", dpi=80)
plt.close()
print("Part 3 plot saved.")



print(f"{'Model':<20} {'D':<4} {'Activation':<12} {'Optimizer':<12} {'Params':<8} {'TrainAcc':<10} {'ValAcc':<10} {'TestAcc'}")
print("-" * 88)

for r in master_results:
    print(f"{r['Model']:<20} {r['Depth']:<4} {r['Activation']:<12} {r['Optimizer']:<12} "
          f"{r['Parameters']:<8} {r['Train Acc']:<10} {r['Val Acc']:<10} {r['Test Acc']}")

# Part 2 and Part 3 rows
part2_rows = [
    ("Dense-Baseline",   2, "relu", "sgd",      p_dense_img,      round(h_dense_img['train_acc'][-1],4), round(h_dense_img['val_acc'][-1],4), round(dense_test_acc,4)),
    ("CNN-WithPool",     1, "relu", "sgd",       cnn_total_params, round(h_pool['train_acc'][-1],4),      round(h_pool['val_acc'][-1],4),      round(acc_pool,4)),
    ("CNN-NoPool",       1, "relu", "sgd",       "-",              round(h_nopool['train_acc'][-1],4),    round(h_nopool['val_acc'][-1],4),    round(acc_nopool,4)),
    ("CNN-Dropout",      1, "relu", "sgd",       "-",              round(h_drop['train_acc'][-1],4),      round(h_drop['val_acc'][-1],4),      round(acc_drop,4)),
    ("CNN-SGD",          1, "relu", "sgd",       cnn_total_params, round(h_sgd['train_acc'][-1],4),       round(h_sgd['val_acc'][-1],4),       round(acc_sgd,4)),
    ("CNN-Momentum",     1, "relu", "momentum",  cnn_total_params, round(h_mom['train_acc'][-1],4),       round(h_mom['val_acc'][-1],4),       round(acc_mom,4)),
    ("CNN-Adam",         1, "relu", "adam",      cnn_total_params, round(h_adam['train_acc'][-1],4),      round(h_adam['val_acc'][-1],4),      round(acc_adam,4)),
]
for r in part2_rows:
    print(f"{str(r[0]):<20} {str(r[1]):<4} {str(r[2]):<12} {str(r[3]):<12} "
          f"{str(r[4]):<8} {str(r[5]):<10} {str(r[6]):<10} {str(r[7])}")

In [1]:
# Analysis Answers PART 1

# Gradient norms comparison explanation
# In deep sigmoid networks the gradient becomes very small as it propagates
# backward through many layers. This is called the vanishing gradient problem.
# For the 10-layer network the first layer receives extremely small gradients,
# so weight updates are very slow. In contrast, the 2-layer network has much
# shorter gradient paths so gradients remain larger and learning is easier.
# This is why deep sigmoid networks train slower and may fail to converge.

# Increasing depth does not always improve validation performance.
# Deeper networks have higher representational capacity but they are also
# harder to train due to optimization difficulties and gradient instability.
# If gradients vanish or explode, deeper networks may perform worse than
# shallower networks even though they have more parameters.

# Test performance may initially improve with depth because deeper networks
# can learn more complex representations. However after a certain point,
# increasing depth can reduce test accuracy due to overfitting or unstable
# training caused by vanishing gradients.

# Yes, sigmoid networks degrade faster with depth.
# The derivative of sigmoid is always less than 1, so during backpropagation
# gradients shrink exponentially across many layers.
# This leads to vanishing gradients and very slow learning in early layers.
# ReLU networks suffer less from this problem.

# Yes. Optimizer choice has a larger impact on deep networks.
# Deep networks require stable gradient updates across many layers.
# Optimizers like Momentum help accelerate learning and reduce oscillations,
# which is especially helpful when gradients become small or noisy.

# Validation performance usually provides a good estimate of test performance
# because it measures generalization on unseen data.
# However it is not always perfect because the validation set is still used
# during model selection. Small dataset variations may cause differences
# between validation and final test accuracy.

In [ ]:
# Analysis Answers PART 2

# Dense networks scale poorly with image size because every input neuron is
# connected to every hidden neuron. This causes the number of parameters to
# grow very quickly as input size increases.
# CNNs use local receptive fields and parameter sharing. The same convolution
# filter is applied across the entire image, so the number of parameters
# depends only on filter size and number of filters rather than image size.
# This makes CNNs much more efficient for image data.

# CNNs usually generalize better than dense networks on image data because
# convolution layers capture spatial patterns such as edges and shapes.
# Dense networks ignore spatial structure and treat pixels independently,
# which makes them less efficient for image recognition tasks.

# Pooling often improves test performance because it reduces spatial
# resolution while keeping important features. This introduces translation
# invariance and reduces the number of parameters, which helps prevent
# overfitting and improves generalization.

# Yes. Dropout reduces overfitting by randomly disabling neurons during
# training. This forces the network to learn more robust feature
# representations and prevents co-adaptation of neurons. As a result the
# gap between validation and test performance often decreases.

# A larger number of parameters increases model capacity, allowing the model
# to fit more complex patterns. However too many parameters can cause
# overfitting, which reduces test accuracy. Good performance depends on
# balancing model capacity with generalization.

# Yes. CNNs scale much better with larger image sizes because convolution
# filters are shared across spatial locations. Increasing image resolution
# does not increase the number of convolution parameters significantly,
# while dense networks would require a huge increase in parameters.

In [3]:
# Analysis Answers PART 3

# SGD usually converges slowly because it follows the gradient direction
# directly and may oscillate in narrow valleys of the loss surface.
# Momentum accelerates convergence by accumulating past gradients, allowing
# faster movement along consistent directions and reducing oscillations.
# Adam generally converges the fastest because it adapts learning rates for
# each parameter using first and second moment estimates of gradients.

# SGD may show noisy and unstable loss curves due to high sensitivity to
# gradient fluctuations. Momentum improves stability by smoothing updates.
# Adam typically provides the most stable training because it uses adaptive
# learning rates and momentum together.

# The master result table summarizes how architecture depth, activation
# function, and optimizer choice affect model performance. It allows direct
# comparison of parameter counts, training accuracy, validation accuracy,
# and test accuracy across all experiments.